            # VisionForge: Trustworthy PyTorch Visual Inspection

            **Custom CNN baseline, transfer learning, calibration, Grad-CAM, corruption stress tests, abstention and deployable inference**

            **Author:** Jorgo Luka  
            **Portfolio level:** Flagship / interview-ready AI product  
            **Format:** Standalone Google Colab notebook  
            **Product:** A field inspection assistant that classifies bean-leaf disease and escalates uncertain images to an agronomist.

            > This notebook is deliberately built as a small production system, not a tutorial.
            > It uses a real public dataset, establishes baselines, trains or applies modern models,
            > evaluates failure modes, converts scores into a decision, and exposes an interactive demo.

            ## Recruiter 30-second scan

            | Signal | Evidence in this notebook |
            |---|---|
            | Computer vision | Custom residual CNN, image augmentation, EfficientNet transfer learning and controlled fine-tuning |
| Trustworthiness | Calibration, uncertainty abstention, slice metrics and image-corruption stress tests |
| Explainability | Grad-CAM heatmaps paired with explicit limitations—not presented as causal proof |
| Deployment | TorchScript/ONNX export, latency benchmark, image contract and Gradio application |
| Engineering | Deterministic splits, mixed precision, early stopping, tests and release manifest |

            **Real dataset:** Makerere Beans: 1,295 real field images labelled healthy, angular leaf spot or bean rust.  
            **Core stack:** PyTorch · torchvision · Hugging Face Datasets · scikit-learn · Grad-CAM · ONNX · Gradio  
            **Decision:** Auto-classify a leaf or abstain for human review based on calibrated uncertainty.

            ### Honest scope

            The system follows frontier-team disciplines—data contracts, baselines, ablations,
            calibration, safety tests, reproducibility, evaluation and deployment—but is sized for
            a free Colab GPU. It does **not** claim frontier-model compute or production certification.
            Every metric shown after execution is evidence from this run; no result is pre-claimed.


## How to run

1. Open this file in Google Colab and select **Runtime → Change runtime type → T4 GPU**.
2. Run the environment cell once. If Colab asks, restart the runtime after installation.
3. Keep `DEMO_MODE = True` for a quick end-to-end proof; switch it off for the stronger run.
4. Leave `LAUNCH_APP = False` during training, then set it to `True` for the live demo.
5. Download the generated model card, metrics, figures and manifest for the GitHub release.

The notebook never requires a private API key. Public model and dataset downloads require
an internet connection. Large downloads are cached by Colab for the current runtime.


## Data provenance and technical references

- **Dataset:** [Makerere AI Lab Beans dataset](https://huggingface.co/datasets/AI-Lab-Makerere/beans), 1,295 images across fixed train/validation/test splits and three classes. The hub metadata lists an MIT licence; the dataset card does not fully document collection and annotation, so production use would require additional provenance review.
- **Original dataset citation:** Makerere AI Lab, *Bean disease dataset* (2020), [source repository](https://github.com/AI-Lab-Makerere/ibean/).
- **Method reference:** [PyTorch transfer learning for computer vision](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html).

The notebook loads only the public dataset and pretrained ImageNet weights. It does not embed or redistribute the source images.


## System architecture

```mermaid
flowchart TD
    A[Leaf photograph] --> B[Image contract]
    B --> C[Custom residual CNN]
    B --> D[EfficientNet transfer model]
    C --> E[Controlled benchmark]
    D --> E
    E --> F[Calibration and abstention]
    F --> G[Disease class and Grad-CAM]
    F --> H[Agronomist review]
```

The business output is not merely a class. It is a triage decision: automate a sufficiently
confident prediction or route the case to a qualified human. In a real deployment, crop
treatment must never be prescribed from this model alone.


## 0. Environment and deterministic experiment configuration


In [1]:
!pip -q install "datasets>=3.0" "evaluate>=0.4" "gradio>=5,<7" \
    "onnx>=1.16" "onnxruntime>=1.19" "torchmetrics>=1.4"


In [2]:
from __future__ import annotations

import contextlib
import dataclasses
import datetime as dt
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import statistics
import sys
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Iterable, Iterator, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


def seed_everything(seed: int = 42) -> None:
    """Seed Python and NumPy; framework seeds are set in their own sections."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


@contextlib.contextmanager
def timer(label: str) -> Iterator[None]:
    started = time.perf_counter()
    print(f"▶ {label}")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - started
        print(f"✓ {label}: {elapsed:,.2f}s")


def human_bytes(value: int | float) -> str:
    size = float(value)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(size) < 1024.0:
            return f"{size:,.1f} {unit}"
        size /= 1024.0
    return f"{size:,.1f} PB"


def save_json(payload: Mapping[str, Any], path: str | Path) -> Path:
    output = Path(path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    return output


def sha256_file(path: str | Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def bootstrap_interval(
    values: Sequence[float],
    statistic: Callable[[np.ndarray], float] = np.mean,
    confidence: float = 0.95,
    rounds: int = 1_000,
    seed: int = 42,
) -> tuple[float, float, float]:
    array = np.asarray(values, dtype=float)
    array = array[np.isfinite(array)]
    if array.size == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    estimates = np.empty(rounds, dtype=float)
    for index in range(rounds):
        sample = rng.choice(array, size=len(array), replace=True)
        estimates[index] = statistic(sample)
    alpha = (1.0 - confidence) / 2.0
    low, high = np.quantile(estimates, [alpha, 1.0 - alpha])
    return float(statistic(array)), float(low), float(high)


def package_versions(names: Sequence[str]) -> dict[str, str]:
    versions: dict[str, str] = {}
    for name in names:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = "not-installed"
    return versions


def runtime_report(packages: Sequence[str]) -> pd.DataFrame:
    rows = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "processor": platform.processor() or "unknown",
        "utc_time": dt.datetime.now(dt.timezone.utc).isoformat(),
        **package_versions(packages),
    }
    return pd.DataFrame(rows.items(), columns=["component", "version"])


def latency_summary(function: Callable[[], Any], repeats: int = 30, warmup: int = 3) -> dict[str, float]:
    for _ in range(warmup):
        function()
    timings = []
    for _ in range(repeats):
        started = time.perf_counter()
        function()
        timings.append((time.perf_counter() - started) * 1_000)
    return {
        "mean_ms": float(np.mean(timings)),
        "p50_ms": float(np.quantile(timings, 0.50)),
        "p95_ms": float(np.quantile(timings, 0.95)),
        "max_ms": float(np.max(timings)),
    }


def assert_finite(name: str, values: Any) -> None:
    array = np.asarray(values)
    if not np.isfinite(array).all():
        raise AssertionError(f"{name} contains non-finite values")


def make_manifest(
    project: str,
    config: Mapping[str, Any],
    metrics: Mapping[str, Any],
    artifacts: Sequence[str | Path],
) -> dict[str, Any]:
    files = []
    for artifact in artifacts:
        path = Path(artifact)
        if path.exists() and path.is_file():
            files.append({
                "path": str(path),
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            })
    return {
        "project": project,
        "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "config": dict(config),
        "metrics": dict(metrics),
        "artifacts": files,
    }


seed_everything(42)


In [3]:
import copy
import io
from collections import OrderedDict

import gradio as gr
import onnxruntime as ort
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0


@dataclass(frozen=True)
class VisionConfig:
    seed: int = 42
    demo_mode: bool = True
    dataset_id: str = "AI-Lab-Makerere/beans"
    image_size: int = 224
    batch_size: int = 32
    scratch_epochs: int = 4
    warmup_epochs: int = 2
    finetune_epochs: int = 3
    warmup_lr: float = 3e-3
    finetune_lr: float = 2e-4
    weight_decay: float = 1e-4
    patience: int = 2
    demo_train_limit: int = 700
    demo_eval_limit: int = 130
    target_review_rate: float = 0.10
    launch_app: bool = False
    artifact_dir: str = "visionforge_artifacts"


CFG = VisionConfig()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACT_DIR = Path(CFG.artifact_dir)
ARTIFACT_DIR.mkdir(exist_ok=True)
seed_everything(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

print(asdict(CFG))
print("device:", DEVICE)
display(runtime_report([
    "torch", "torchvision", "datasets", "scikit-learn",
    "onnx", "onnxruntime", "gradio",
]))


2026-08-12 13:20:48.816052571 [W:onnxruntime:Default, device_discovery.cc:134 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/ACPI0004:00/VMBUS:00/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


{'seed': 42, 'demo_mode': True, 'dataset_id': 'AI-Lab-Makerere/beans', 'image_size': 224, 'batch_size': 32, 'scratch_epochs': 4, 'warmup_epochs': 2, 'finetune_epochs': 3, 'warmup_lr': 0.003, 'finetune_lr': 0.0002, 'weight_decay': 0.0001, 'patience': 2, 'demo_train_limit': 700, 'demo_eval_limit': 130, 'target_review_rate': 0.1, 'launch_app': False, 'artifact_dir': 'visionforge_artifacts'}
device: cpu


,component,version
0,python,3.11.15
1,platform,Linux-6.17.0-1020-azure-x86_64-with-glibc2.39
2,processor,x86_64
3,utc_time,2026-08-12T13:20:51.891344+00:00
4,torch,2.13.0+cpu
5,torchvision,0.28.0+cpu
6,datasets,5.0.1
7,scikit-learn,1.9.0
8,onnx,1.22.0
9,onnxruntime,1.28.0


## 1. Real image data, visual audit and input contract


In [4]:
with timer("download Makerere Beans dataset"):
    raw = load_dataset(CFG.dataset_id)

label_feature = raw["train"].features["labels"] if "labels" in raw["train"].features else raw["train"].features["label"]
LABEL_COLUMN = "labels" if "labels" in raw["train"].column_names else "label"
IMAGE_COLUMN = "image"
CLASS_NAMES = list(label_feature.names)
NUM_CLASSES = len(CLASS_NAMES)


def deterministic_subset(dataset: Any, limit: int | None, seed: int) -> Any:
    if limit is None or limit >= len(dataset):
        return dataset
    rng = np.random.default_rng(seed)
    indices = np.sort(rng.choice(len(dataset), size=limit, replace=False))
    return dataset.select(indices.tolist())


train_raw = deterministic_subset(
    raw["train"],
    CFG.demo_train_limit if CFG.demo_mode else None,
    CFG.seed,
)
valid_raw = deterministic_subset(
    raw["validation"],
    CFG.demo_eval_limit if CFG.demo_mode else None,
    CFG.seed + 1,
)
test_raw = deterministic_subset(
    raw["test"],
    CFG.demo_eval_limit if CFG.demo_mode else None,
    CFG.seed + 2,
)


def image_contract(dataset: Any, split: str) -> pd.DataFrame:
    rows = []
    labels = []
    for index in range(len(dataset)):
        item = dataset[index]
        image = item[IMAGE_COLUMN]
        label = int(item[LABEL_COLUMN])
        labels.append(label)
        rows.append({
            "split": split,
            "index": index,
            "width": image.width,
            "height": image.height,
            "mode": image.mode,
            "label": label,
            "class_name": CLASS_NAMES[label],
        })
    frame = pd.DataFrame(rows)
    assert frame["label"].between(0, NUM_CLASSES - 1).all()
    assert frame[["width", "height"]].gt(0).all().all()
    assert set(frame["class_name"]).issubset(set(CLASS_NAMES))
    return frame


audit = pd.concat([
    image_contract(train_raw, "train"),
    image_contract(valid_raw, "validation"),
    image_contract(test_raw, "test"),
], ignore_index=True)
display(pd.crosstab(audit["split"], audit["class_name"], margins=True))
display(audit.groupby("split")[["width", "height"]].describe().round(1))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for axis, index in zip(axes.flat, np.linspace(0, len(train_raw) - 1, 8, dtype=int)):
    item = train_raw[int(index)]
    axis.imshow(item[IMAGE_COLUMN])
    axis.set_title(CLASS_NAMES[int(item[LABEL_COLUMN])])
    axis.axis("off")
plt.suptitle("Real field images—inspect lighting, scale and background variation", y=1.01)
plt.tight_layout()
sample_figure = ARTIFACT_DIR / "data_samples.png"
plt.savefig(sample_figure, dpi=160, bbox_inches="tight")
plt.show()


▶ download Makerere Beans dataset


README.md:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  144MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.5MB            

...

Generating validation split:   0%|          | 0/133 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

✓ download Makerere Beans dataset: 6.88s


class_name,angular_leaf_spot,bean_rust,healthy,All
split,,,,
test,43,43,42,128
train,242,237,221,700
validation,43,43,44,130
All,328,323,307,958


width                                                height                                               
            count   mean  std    min    25%    50%    75%    max  count   mean  std    min    25%    50%    75%    max
split                                                                                                                 
test        128.0  500.0  0.0  500.0  500.0  500.0  500.0  500.0  128.0  500.0  0.0  500.0  500.0  500.0  500.0  500.0
train       700.0  500.0  0.0  500.0  500.0  500.0  500.0  500.0  700.0  500.0  0.0  500.0  500.0  500.0  500.0  500.0
validation  130.0  500.0  0.0  500.0  500.0  500.0  500.0  500.0  130.0  500.0  0.0  500.0  500.0  500.0  500.0  500.0

## 2. Augmentation policy and leakage-safe data loaders


In [5]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = T.Compose([
    T.RandomResizedCrop(CFG.image_size, scale=(0.72, 1.0), ratio=(0.85, 1.15)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.15),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.12, hue=0.03),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.15, scale=(0.02, 0.10), ratio=(0.5, 2.0)),
])
eval_transform = T.Compose([
    T.Resize(int(CFG.image_size * 1.12)),
    T.CenterCrop(CFG.image_size),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class HFDatasetAdapter(Dataset):
    def __init__(self, dataset: Any, transform: Callable[[Image.Image], torch.Tensor]):
        self.dataset = dataset
        self.transform = transform

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        item = self.dataset[int(index)]
        image = item[IMAGE_COLUMN].convert("RGB")
        label = int(item[LABEL_COLUMN])
        return self.transform(image), label


train_dataset = HFDatasetAdapter(train_raw, train_transform)
valid_dataset = HFDatasetAdapter(valid_raw, eval_transform)
test_dataset = HFDatasetAdapter(test_raw, eval_transform)

generator = torch.Generator().manual_seed(CFG.seed)
loader_kwargs = {
    "batch_size": CFG.batch_size,
    "num_workers": 2,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": True,
}
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_kwargs)
valid_loader = DataLoader(valid_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

batch_images, batch_labels = next(iter(train_loader))
print("batch:", batch_images.shape, batch_labels.shape)
assert batch_images.shape[1:] == (3, CFG.image_size, CFG.image_size)
assert torch.isfinite(batch_images).all()


batch: torch.Size([32, 3, 224, 224]) torch.Size([32])


## 3. Transfer learning with a staged optimisation policy

First train only the classifier head to avoid destroying pretrained visual features. Then
unfreeze the final feature blocks at a lower learning rate. The validation loss controls
checkpoint selection; the test set remains untouched until the final evaluation.


In [6]:
def build_model(num_classes: int, pretrained: bool = True) -> nn.Module:
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    network = efficientnet_b0(weights=weights)
    input_features = network.classifier[1].in_features
    network.classifier = nn.Sequential(
        nn.Dropout(p=0.30),
        nn.Linear(input_features, 256),
        nn.SiLU(),
        nn.Dropout(p=0.20),
        nn.Linear(256, num_classes),
    )
    return network


model = build_model(NUM_CLASSES).to(DEVICE)
for parameter in model.features.parameters():
    parameter.requires_grad = False

train_counts = audit.loc[audit.split == "train", "label"].value_counts().sort_index()
class_weights = len(train_raw) / (NUM_CLASSES * train_counts.to_numpy())
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.05)


def count_parameters(network: nn.Module) -> dict[str, int]:
    return {
        "total": sum(parameter.numel() for parameter in network.parameters()),
        "trainable": sum(parameter.numel() for parameter in network.parameters() if parameter.requires_grad),
    }


display(pd.DataFrame([count_parameters(model)]))
print("class weights:", dict(zip(CLASS_NAMES, class_weights.round(3))))


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/runner/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


  0%|          | 0.00/20.5M [00:00<?, ?B/s]

100%|██████████| 20.5M/20.5M [00:00<00:00, 270MB/s]

,total,trainable
0,4336255,328707


class weights: {'angular_leaf_spot': np.float64(0.964), 'bean_rust': np.float64(0.985), 'healthy': np.float64(1.056)}


In [7]:
@dataclass
class EpochResult:
    loss: float
    accuracy: float
    macro_f1: float
    samples: int


def run_epoch(
    network: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None = None,
    scaler: torch.cuda.amp.GradScaler | None = None,
) -> EpochResult:
    training = optimizer is not None
    network.train(training)
    losses, labels_all, predictions_all = [], [], []
    autocast_enabled = DEVICE.type == "cuda"
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=autocast_enabled):
            logits = network(images)
            loss = criterion(logits, labels)
        if training:
            if scaler is not None and autocast_enabled:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
                optimizer.step()
        losses.append(float(loss.detach().cpu()) * len(labels))
        labels_all.extend(labels.detach().cpu().tolist())
        predictions_all.extend(logits.argmax(dim=1).detach().cpu().tolist())
    return EpochResult(
        loss=float(sum(losses) / len(labels_all)),
        accuracy=float(accuracy_score(labels_all, predictions_all)),
        macro_f1=float(f1_score(labels_all, predictions_all, average="macro")),
        samples=len(labels_all),
    )


def fit_stage(
    network: nn.Module,
    epochs: int,
    learning_rate: float,
    stage: str,
) -> tuple[nn.Module, list[dict[str, Any]]]:
    optimizer = torch.optim.AdamW(
        [parameter for parameter in network.parameters() if parameter.requires_grad],
        lr=learning_rate,
        weight_decay=CFG.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    best_state = copy.deepcopy(network.state_dict())
    best_loss = float("inf")
    stale_epochs = 0
    history = []
    for epoch in range(1, epochs + 1):
        train_result = run_epoch(network, train_loader, optimizer=optimizer, scaler=scaler)
        with torch.inference_mode():
            valid_result = run_epoch(network, valid_loader)
        row = {
            "stage": stage,
            "epoch": epoch,
            "learning_rate": optimizer.param_groups[0]["lr"],
            **{f"train_{key}": value for key, value in asdict(train_result).items()},
            **{f"valid_{key}": value for key, value in asdict(valid_result).items()},
        }
        history.append(row)
        print(
            f"{stage} {epoch:02d}/{epochs} | "
            f"train loss {train_result.loss:.4f} | "
            f"valid loss {valid_result.loss:.4f} | "
            f"valid F1 {valid_result.macro_f1:.3f}"
        )
        if valid_result.loss < best_loss - 1e-4:
            best_loss = valid_result.loss
            best_state = copy.deepcopy(network.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
        scheduler.step()
        if stale_epochs >= CFG.patience:
            print("early stopping")
            break
    network.load_state_dict(best_state)
    return network, history


with timer("classifier-head warmup"):
    model, warmup_history = fit_stage(
        model,
        epochs=CFG.warmup_epochs if CFG.demo_mode else 5,
        learning_rate=CFG.warmup_lr,
        stage="head",
    )

for block in list(model.features.children())[-3:]:
    for parameter in block.parameters():
        parameter.requires_grad = True

with timer("selective backbone fine-tuning"):
    model, finetune_history = fit_stage(
        model,
        epochs=CFG.finetune_epochs if CFG.demo_mode else 12,
        learning_rate=CFG.finetune_lr,
        stage="fine_tune",
    )

training_history = pd.DataFrame(warmup_history + finetune_history)
display(training_history.tail())


▶ classifier-head warmup


head 01/2 | train loss 0.8194 | valid loss 0.5128 | valid F1 0.840


head 02/2 | train loss 0.6267 | valid loss 0.4835 | valid F1 0.844
✓ classifier-head warmup: 38.64s
▶ selective backbone fine-tuning


fine_tune 01/3 | train loss 0.5465 | valid loss 0.3861 | valid F1 0.895


fine_tune 02/3 | train loss 0.4026 | valid loss 0.3948 | valid F1 0.904


fine_tune 03/3 | train loss 0.3646 | valid loss 0.3861 | valid F1 0.913
early stopping
✓ selective backbone fine-tuning: 72.06s


,stage,epoch,learning_rate,train_loss,train_accuracy,train_macro_f1,train_samples,valid_loss,valid_accuracy,valid_macro_f1,valid_samples
0,head,1,0.00300,0.819382,0.648571,0.650496,700,0.512787,0.846154,0.840293,130
1,head,2,0.00150,0.626684,0.778571,0.779504,700,0.483496,0.846154,0.843838,130
2,fine_tune,1,0.00020,0.546521,0.827143,0.828849,700,0.386120,0.900000,0.895431,130
3,fine_tune,2,0.00015,0.402638,0.880000,0.880417,700,0.394810,0.907692,0.904400,130
4,fine_tune,3,0.00005,0.364602,0.927143,0.927736,700,0.386101,0.915385,0.912644,130


## 4. Test metrics, per-class errors and calibration


In [8]:
def collect_predictions(
    network: nn.Module,
    loader: DataLoader,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    network.eval()
    logits_all, labels_all = [], []
    with torch.inference_mode():
        for images, labels in loader:
            logits = network(images.to(DEVICE, non_blocking=True))
            logits_all.append(logits.cpu())
            labels_all.append(labels.cpu())
    logits_array = torch.cat(logits_all).numpy()
    labels_array = torch.cat(labels_all).numpy()
    probabilities = torch.softmax(torch.from_numpy(logits_array), dim=1).numpy()
    return logits_array, probabilities, labels_array


valid_logits, valid_probabilities, valid_labels = collect_predictions(model, valid_loader)
test_logits, test_probabilities, test_labels = collect_predictions(model, test_loader)


class TemperatureScaler(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self) -> torch.Tensor:
        return self.log_temperature.exp().clamp(0.05, 20.0)

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.temperature


calibrator = TemperatureScaler().to(DEVICE)
validation_logits_tensor = torch.tensor(valid_logits, dtype=torch.float32, device=DEVICE)
validation_labels_tensor = torch.tensor(valid_labels, dtype=torch.long, device=DEVICE)
calibration_optimizer = torch.optim.LBFGS(calibrator.parameters(), lr=0.1, max_iter=50)


def calibration_closure() -> torch.Tensor:
    calibration_optimizer.zero_grad()
    loss = F.cross_entropy(calibrator(validation_logits_tensor), validation_labels_tensor)
    loss.backward()
    return loss


calibration_optimizer.step(calibration_closure)
temperature = float(calibrator.temperature.detach().cpu())
calibrated_test_probabilities = torch.softmax(
    torch.tensor(test_logits) / temperature,
    dim=1,
).numpy()


def expected_calibration_error(
    probabilities: np.ndarray,
    labels: np.ndarray,
    bins: int = 12,
) -> tuple[float, pd.DataFrame]:
    confidence = probabilities.max(axis=1)
    prediction = probabilities.argmax(axis=1)
    correct = prediction == labels
    boundaries = np.linspace(0.0, 1.0, bins + 1)
    rows, ece = [], 0.0
    for lower, upper in zip(boundaries[:-1], boundaries[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if not mask.any():
            continue
        accuracy = float(correct[mask].mean())
        average_confidence = float(confidence[mask].mean())
        weight = float(mask.mean())
        ece += weight * abs(accuracy - average_confidence)
        rows.append({
            "lower": lower,
            "upper": upper,
            "count": int(mask.sum()),
            "accuracy": accuracy,
            "confidence": average_confidence,
            "gap": average_confidence - accuracy,
        })
    return float(ece), pd.DataFrame(rows)


raw_ece, raw_reliability = expected_calibration_error(test_probabilities, test_labels)
calibrated_ece, calibrated_reliability = expected_calibration_error(calibrated_test_probabilities, test_labels)
test_predictions = calibrated_test_probabilities.argmax(axis=1)

headline_metrics = {
    "accuracy": float(accuracy_score(test_labels, test_predictions)),
    "balanced_accuracy": float(balanced_accuracy_score(test_labels, test_predictions)),
    "macro_f1": float(f1_score(test_labels, test_predictions, average="macro")),
    "negative_log_likelihood": float(log_loss(test_labels, calibrated_test_probabilities)),
    "ece_before": raw_ece,
    "ece_after": calibrated_ece,
    "temperature": temperature,
}
display(pd.DataFrame([headline_metrics]).T.rename(columns={0: "value"}).style.format("{:.4f}"))

report = pd.DataFrame(classification_report(
    test_labels,
    test_predictions,
    target_names=CLASS_NAMES,
    output_dict=True,
)).T
display(report.style.format("{:.3f}"))


,value
accuracy,0.8594
balanced_accuracy,0.8603
macro_f1,0.8575
negative_log_likelihood,0.3112
ece_before,0.0578
ece_after,0.0554
temperature,0.7659


,precision,recall,f1-score,support
angular_leaf_spot,0.886,0.721,0.795,43.000
bean_rust,0.792,0.884,0.835,43.000
healthy,0.911,0.976,0.943,42.000
accuracy,0.859,0.859,0.859,0.859
macro avg,0.863,0.860,0.858,128.000
weighted avg,0.862,0.859,0.857,128.000


## 3B. Custom residual CNN baseline versus transfer learning

A transfer-learning result is much more convincing when compared with a controlled model trained from random initialisation. The baseline below uses convolution, batch normalisation, nonlinear activation, residual connections, downsampling, global average pooling and dropout. It sees the **same split, augmentation policy, loss weighting and early-stopping protocol** as EfficientNet.

This is an engineering benchmark rather than a claim that the larger pretrained model is always superior. The comparison makes the value of learned ImageNet representations measurable on this small field dataset.


In [9]:
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.skip = (
            nn.Identity()
            if in_channels == out_channels and stride == 1
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        )
        self.activation = nn.GELU()

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.activation(self.main(inputs) + self.skip(inputs))


class ScratchResidualCNN(nn.Module):
    # Compact CNN trained from random initialisation for an honest baseline.
    def __init__(self, num_classes: int):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.features = nn.Sequential(
            ResidualConvBlock(32, 48, stride=2),
            ResidualConvBlock(48, 64, stride=2),
            ResidualConvBlock(64, 96, stride=2),
            ResidualConvBlock(96, 128, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, num_classes),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        features = self.features(self.stem(inputs))
        return self.classifier(self.pool(features))


scratch_model = ScratchResidualCNN(NUM_CLASSES).to(DEVICE)
with timer("train custom residual CNN baseline"):
    scratch_model, scratch_history = fit_stage(
        scratch_model,
        epochs=CFG.scratch_epochs if CFG.demo_mode else 12,
        learning_rate=8e-4,
        stage="scratch_cnn",
    )

scratch_logits, scratch_probabilities, scratch_labels = collect_predictions(
    scratch_model,
    test_loader,
)
scratch_predictions = scratch_probabilities.argmax(axis=1)
scratch_metrics = {
    "accuracy": float(accuracy_score(scratch_labels, scratch_predictions)),
    "balanced_accuracy": float(balanced_accuracy_score(scratch_labels, scratch_predictions)),
    "macro_f1": float(f1_score(scratch_labels, scratch_predictions, average="macro")),
    "negative_log_likelihood": float(log_loss(scratch_labels, scratch_probabilities)),
}

benchmark_table = pd.DataFrame([
    {
        "model": "ScratchResidualCNN",
        "initialisation": "random",
        "parameters": count_parameters(scratch_model)["total"],
        **scratch_metrics,
    },
    {
        "model": "EfficientNet-B0",
        "initialisation": "ImageNet transfer",
        "parameters": count_parameters(model)["total"],
        **{key: headline_metrics[key] for key in scratch_metrics},
    },
]).sort_values("macro_f1", ascending=False)

training_history = pd.concat(
    [training_history, pd.DataFrame(scratch_history)],
    ignore_index=True,
)
scratch_checkpoint_path = ARTIFACT_DIR / "visionforge_scratch_cnn_state_dict.pt"
torch.save({
    "state_dict": scratch_model.state_dict(),
    "class_names": CLASS_NAMES,
    "config": asdict(CFG),
    "metrics": scratch_metrics,
}, scratch_checkpoint_path)
benchmark_csv = ARTIFACT_DIR / "model_benchmark.csv"
benchmark_table.to_csv(benchmark_csv, index=False)

display(benchmark_table.style.format({
    "parameters": "{:,.0f}",
    "accuracy": "{:.3f}",
    "balanced_accuracy": "{:.3f}",
    "macro_f1": "{:.3f}",
    "negative_log_likelihood": "{:.3f}",
}))

fig, ax = plt.subplots(figsize=(9, 4.5))
benchmark_long = benchmark_table.melt(
    id_vars=["model"],
    value_vars=["accuracy", "balanced_accuracy", "macro_f1"],
    var_name="metric",
    value_name="score",
)
sns.barplot(data=benchmark_long, x="metric", y="score", hue="model", ax=ax)
ax.set(title="Controlled CNN benchmark", ylim=(0, 1), ylabel="test score")
ax.legend(loc="lower right")
plt.tight_layout()
benchmark_figure = ARTIFACT_DIR / "cnn_transfer_benchmark.png"
plt.savefig(benchmark_figure, dpi=170, bbox_inches="tight")
plt.show()


▶ train custom residual CNN baseline


scratch_cnn 01/4 | train loss 0.9532 | valid loss 1.0996 | valid F1 0.436


scratch_cnn 02/4 | train loss 0.8646 | valid loss 0.8038 | valid F1 0.656


scratch_cnn 03/4 | train loss 0.7894 | valid loss 0.6850 | valid F1 0.748


scratch_cnn 04/4 | train loss 0.7296 | valid loss 0.6303 | valid F1 0.755
✓ train custom residual CNN baseline: 55.52s


,model,initialisation,parameters,accuracy,balanced_accuracy,macro_f1,negative_log_likelihood
1,EfficientNet-B0,ImageNet transfer,"4,336,255",0.859,0.860,0.858,0.311
0,ScratchResidualCNN,random,"523,267",0.719,0.720,0.707,0.607


In [10]:
confusion = confusion_matrix(test_labels, test_predictions, normalize="true")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(
    confusion,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=axes[0],
)
axes[0].set(title="Row-normalised confusion matrix", xlabel="predicted", ylabel="actual")
for frame, label, colour in [
    (raw_reliability, "before", "#ef4444"),
    (calibrated_reliability, "after", "#2563eb"),
]:
    axes[1].plot(frame["confidence"], frame["accuracy"], marker="o", label=label, color=colour)
axes[1].plot([0, 1], [0, 1], "--", color="black", alpha=0.6)
axes[1].set(title="Reliability diagram", xlabel="confidence", ylabel="accuracy", xlim=(0, 1), ylim=(0, 1))
axes[1].legend()
history_plot = training_history.copy()
axes[2].plot(range(len(history_plot)), history_plot["train_loss"], marker="o", label="train")
axes[2].plot(range(len(history_plot)), history_plot["valid_loss"], marker="o", label="validation")
axes[2].set(title="Loss by training stage", xlabel="epoch", ylabel="cross-entropy")
axes[2].legend()
plt.tight_layout()
diagnostics_figure = ARTIFACT_DIR / "model_diagnostics.png"
plt.savefig(diagnostics_figure, dpi=170, bbox_inches="tight")
plt.show()


## 5. Selective prediction: automate or escalate


In [11]:
calibrated_valid_probabilities = torch.softmax(
    torch.tensor(valid_logits) / temperature,
    dim=1,
).numpy()
validation_confidence = calibrated_valid_probabilities.max(axis=1)
confidence_threshold = float(np.quantile(validation_confidence, CFG.target_review_rate))


def selective_metrics(
    probabilities: np.ndarray,
    labels: np.ndarray,
    threshold: float,
) -> dict[str, float]:
    confidence = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    accepted = confidence >= threshold
    coverage = float(accepted.mean())
    selective_accuracy = float((predictions[accepted] == labels[accepted]).mean()) if accepted.any() else float("nan")
    errors_escalated = float((~accepted & (predictions != labels)).sum() / max((predictions != labels).sum(), 1))
    return {
        "threshold": threshold,
        "coverage": coverage,
        "review_rate": 1.0 - coverage,
        "selective_accuracy": selective_accuracy,
        "errors_escalated": errors_escalated,
    }


threshold_grid = np.linspace(0.34, 0.98, 50)
selective_curve = pd.DataFrame([
    selective_metrics(calibrated_test_probabilities, test_labels, threshold)
    for threshold in threshold_grid
])
selected_policy = selective_metrics(
    calibrated_test_probabilities,
    test_labels,
    confidence_threshold,
)
display(pd.DataFrame([selected_policy]).style.format("{:.3f}"))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(selective_curve["coverage"], selective_curve["selective_accuracy"], color="#7c3aed")
ax.scatter(selected_policy["coverage"], selected_policy["selective_accuracy"], s=100, color="#f59e0b", label="chosen policy")
ax.set(title="Accuracy–coverage trade-off", xlabel="automated coverage", ylabel="accuracy among automated cases", xlim=(0, 1), ylim=(0, 1))
ax.legend()
plt.tight_layout()
policy_figure = ARTIFACT_DIR / "selective_policy.png"
plt.savefig(policy_figure, dpi=170, bbox_inches="tight")
plt.show()


,threshold,coverage,review_rate,selective_accuracy,errors_escalated
0,0.676,0.891,0.109,0.904,0.389


## 6. Corruption stress tests and operational slices


In [12]:
def corrupt_batch(images: torch.Tensor, corruption: str, severity: float) -> torch.Tensor:
    output = images.clone()
    if corruption == "gaussian_noise":
        output = output + torch.randn_like(output) * severity
    elif corruption == "blur":
        kernel = int(3 + 2 * round(severity * 3))
        if kernel % 2 == 0:
            kernel += 1
        output = TF.gaussian_blur(output, kernel_size=[kernel, kernel], sigma=max(0.1, severity * 3))
    elif corruption == "darkness":
        output = output * max(0.15, 1.0 - severity)
    elif corruption == "occlusion":
        side = int(output.shape[-1] * min(0.50, severity))
        start = (output.shape[-1] - side) // 2
        output[:, :, start : start + side, start : start + side] = 0.0
    else:
        raise ValueError(f"Unknown corruption: {corruption}")
    return output


def evaluate_corruption(corruption: str, severity: float) -> dict[str, Any]:
    logits_all, labels_all = [], []
    model.eval()
    with torch.inference_mode():
        for images, labels in test_loader:
            altered = corrupt_batch(images, corruption, severity).to(DEVICE)
            logits_all.append(model(altered).cpu())
            labels_all.append(labels)
    logits = torch.cat(logits_all) / temperature
    probabilities = torch.softmax(logits, dim=1).numpy()
    labels = torch.cat(labels_all).numpy()
    predictions = probabilities.argmax(axis=1)
    policy = selective_metrics(probabilities, labels, confidence_threshold)
    return {
        "corruption": corruption,
        "severity": severity,
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro")),
        **policy,
    }


stress_plan = [
    ("gaussian_noise", 0.10),
    ("gaussian_noise", 0.20),
    ("blur", 0.35),
    ("blur", 0.70),
    ("darkness", 0.30),
    ("darkness", 0.60),
    ("occlusion", 0.20),
    ("occlusion", 0.35),
]
with timer("run corruption suite"):
    corruption_report = pd.DataFrame([
        evaluate_corruption(name, severity)
        for name, severity in stress_plan
    ])
display(corruption_report.style.format({
    "severity": "{:.2f}", "accuracy": "{:.3f}", "macro_f1": "{:.3f}",
    "coverage": "{:.3f}", "selective_accuracy": "{:.3f}",
}))

fig, ax = plt.subplots(figsize=(11, 5))
sns.scatterplot(
    data=corruption_report,
    x="coverage",
    y="selective_accuracy",
    hue="corruption",
    size="severity",
    sizes=(70, 220),
    ax=ax,
)
ax.set(title="Robustness under realistic image degradation", xlim=(0, 1), ylim=(0, 1))
plt.tight_layout()
stress_figure = ARTIFACT_DIR / "corruption_stress.png"
plt.savefig(stress_figure, dpi=170, bbox_inches="tight")
plt.show()


▶ run corruption suite


✓ run corruption suite: 16.19s


,corruption,severity,accuracy,macro_f1,threshold,coverage,review_rate,selective_accuracy,errors_escalated
0,gaussian_noise,0.10,0.875,0.871,0.675914,0.898,0.101562,0.904,0.312500
1,gaussian_noise,0.20,0.898,0.899,0.675914,0.797,0.203125,0.941,0.538462
2,blur,0.35,0.641,0.630,0.675914,0.648,0.351562,0.735,0.521739
3,blur,0.70,0.422,0.317,0.675914,0.555,0.445312,0.465,0.486486
4,darkness,0.30,0.891,0.890,0.675914,0.836,0.164062,0.953,0.642857
5,darkness,0.60,0.727,0.707,0.675914,0.781,0.218750,0.810,0.457143
6,occlusion,0.20,0.859,0.858,0.675914,0.883,0.117188,0.903,0.388889
7,occlusion,0.35,0.859,0.859,0.675914,0.891,0.109375,0.886,0.277778


## 7. Grad-CAM explanation with limitations


In [13]:
class GradCAM:
    def __init__(self, network: nn.Module, target_layer: nn.Module):
        self.network = network
        self.activations: torch.Tensor | None = None
        self.gradients: torch.Tensor | None = None
        self.forward_handle = target_layer.register_forward_hook(self._capture_activation)
        self.backward_handle = target_layer.register_full_backward_hook(self._capture_gradient)

    def _capture_activation(self, module: nn.Module, inputs: tuple[Any, ...], output: torch.Tensor) -> None:
        self.activations = output.detach()

    def _capture_gradient(
        self,
        module: nn.Module,
        grad_input: tuple[torch.Tensor, ...],
        grad_output: tuple[torch.Tensor, ...],
    ) -> None:
        self.gradients = grad_output[0].detach()

    def __call__(self, image_tensor: torch.Tensor, target_class: int | None = None) -> tuple[np.ndarray, int]:
        self.network.eval()
        self.network.zero_grad(set_to_none=True)
        logits = self.network(image_tensor)
        selected = int(logits.argmax(dim=1).item()) if target_class is None else int(target_class)
        logits[0, selected].backward()
        if self.activations is None or self.gradients is None:
            raise RuntimeError("Grad-CAM hooks did not capture tensors")
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True).relu()
        cam = F.interpolate(cam, size=image_tensor.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), selected

    def close(self) -> None:
        self.forward_handle.remove()
        self.backward_handle.remove()


grad_cam = GradCAM(model, model.features[-1])


def denormalise(tensor: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    image = tensor.cpu() * std + mean
    return image.clamp(0, 1).permute(1, 2, 0).numpy()


demo_tensor, demo_label = test_dataset[0]
heatmap, demo_prediction = grad_cam(demo_tensor.unsqueeze(0).to(DEVICE))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(denormalise(demo_tensor))
axes[0].set_title(f"Actual: {CLASS_NAMES[demo_label]}")
axes[1].imshow(heatmap, cmap="magma")
axes[1].set_title("Grad-CAM activation")
axes[2].imshow(denormalise(demo_tensor))
axes[2].imshow(heatmap, cmap="jet", alpha=0.42)
axes[2].set_title(f"Predicted: {CLASS_NAMES[demo_prediction]}")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
explainability_figure = ARTIFACT_DIR / "gradcam_example.png"
plt.savefig(explainability_figure, dpi=170, bbox_inches="tight")
plt.show()
print("Interpretation guardrail: Grad-CAM shows sensitivity, not causal disease evidence.")


Interpretation guardrail: Grad-CAM shows sensitivity, not causal disease evidence.


## 8. Export, latency and interactive inspection application


In [14]:
model.eval()
checkpoint_path = ARTIFACT_DIR / "visionforge_state_dict.pt"
torch.save({
    "state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "temperature": temperature,
    "confidence_threshold": confidence_threshold,
    "config": asdict(CFG),
}, checkpoint_path)

# Grad-CAM registers backward hooks on `model`. TorchScript deliberately refuses
# to compile modules carrying those hooks. Reconstruct the same architecture without
# pretrained downloads or hooks, then load the trained tensors into that clean instance.
export_model = build_model(NUM_CLASSES, pretrained=False).cpu()
export_model.load_state_dict({
    name: tensor.detach().cpu()
    for name, tensor in model.state_dict().items()
})
export_model.eval()
assert not any(module._backward_hooks for module in export_model.modules())

example_input = torch.randn(1, 3, CFG.image_size, CFG.image_size)
with torch.inference_mode():
    traced_model = torch.jit.trace(export_model, example_input)
torchscript_path = ARTIFACT_DIR / "visionforge_efficientnet.ts"
traced_model.save(str(torchscript_path))

onnx_path = ARTIFACT_DIR / "visionforge_efficientnet.onnx"
torch.onnx.export(
    export_model,
    example_input,
    onnx_path,
    input_names=["image"],
    output_names=["logits"],
    dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

inference_sample = next(iter(test_loader))[0][:1].to(DEVICE)
def timed_inference() -> torch.Tensor:
    with torch.inference_mode():
        return model(inference_sample)

latency = latency_summary(
    timed_inference,
    repeats=50 if DEVICE.type == "cuda" else 15,
    warmup=5,
)
print("PyTorch latency:", latency)
print("checkpoint:", human_bytes(checkpoint_path.stat().st_size))
print("TorchScript:", human_bytes(torchscript_path.stat().st_size))
print("ONNX:", human_bytes(onnx_path.stat().st_size))


/tmp/ipykernel_2353/1890684768.py:29: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0812 13:24:14.479000 2353 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/opt/hostedtoolcache/Python/3.11.15/x64/lib/python3.11/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/hostedtoolcache/Python/3.11.15/x64/lib/python3.11/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/opt/hostedtoolcache/Python/3.11.15/x64/lib/python3.11/site-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/hostedtoolcache/Python/3.11.15/x64/lib/python3.11/site-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.conv

[torch.onnx] Optimize the ONNX graph...


[torch.onnx] Optimize the ONNX graph... ✅


PyTorch latency: {'mean_ms': 20.18291566666524, 'p50_ms': 19.09378900001002, 'p95_ms': 26.727497099983566, 'max_ms': 26.830744999983835}
checkpoint: 16.8 MB
TorchScript: 17.4 MB
ONNX: 641.7 KB


In [15]:
def prepare_pil(image: Image.Image) -> torch.Tensor:
    if image is None:
        raise ValueError("An image is required")
    image = image.convert("RGB")
    if min(image.size) < 32:
        raise ValueError("Image is too small; minimum side is 32 pixels")
    return eval_transform(image)


def predict_leaf(image: Image.Image) -> tuple[str, pd.DataFrame, Image.Image]:
    tensor = prepare_pil(image)
    model.eval()
    with torch.inference_mode():
        logits = model(tensor.unsqueeze(0).to(DEVICE)) / temperature
        probabilities = torch.softmax(logits, dim=1)[0].cpu().numpy()
    predicted = int(probabilities.argmax())
    confidence = float(probabilities[predicted])
    decision = "AUTO-CLASSIFY" if confidence >= confidence_threshold else "HUMAN REVIEW"
    table = pd.DataFrame({
        "class": CLASS_NAMES,
        "probability": probabilities,
    }).sort_values("probability", ascending=False)
    heatmap, _ = grad_cam(tensor.unsqueeze(0).to(DEVICE), predicted)
    base = np.asarray(image.convert("RGB").resize((CFG.image_size, CFG.image_size))).astype(float) / 255.0
    colour = plt.get_cmap("jet")(heatmap)[..., :3]
    overlay = Image.fromarray(np.uint8(np.clip(base * 0.60 + colour * 0.40, 0, 1) * 255))
    summary = (
        f"### {decision}\n"
        f"Prediction: **{CLASS_NAMES[predicted]}**  \n"
        f"Calibrated confidence: **{confidence:.1%}**  \n"
        f"Review threshold: **{confidence_threshold:.1%}**"
    )
    return summary, table, overlay


with gr.Blocks(title="VisionForge") as vision_app:
    gr.Markdown("# VisionForge\nUpload a bean-leaf image for calibrated triage and a sensitivity map.")
    with gr.Row():
        image_input = gr.Image(type="pil", label="Leaf photograph")
        heatmap_output = gr.Image(type="pil", label="Grad-CAM overlay")
    classify_button = gr.Button("Inspect leaf", variant="primary")
    decision_output = gr.Markdown()
    probability_output = gr.Dataframe(label="Calibrated probabilities", interactive=False)
    classify_button.click(
        predict_leaf,
        image_input,
        [decision_output, probability_output, heatmap_output],
    )

if CFG.launch_app:
    vision_app.launch(share=True, debug=False)
else:
    print("App assembled. Set launch_app=True in VisionConfig and rerun this cell.")


App assembled. Set launch_app=True in VisionConfig and rerun this cell.


## 9. Tests, model card and release manifest


In [16]:
def run_unit_tests() -> None:
    assert len(CLASS_NAMES) == NUM_CLASSES == 3
    assert set(audit["split"]) == {"train", "validation", "test"}
    assert batch_images.shape[1:] == (3, CFG.image_size, CFG.image_size)
    assert np.allclose(calibrated_test_probabilities.sum(axis=1), 1.0, atol=1e-5)
    assert 0.0 < temperature < 20.0
    assert 0.0 <= confidence_threshold <= 1.0
    assert 0.0 <= selected_policy["coverage"] <= 1.0
    assert 0.0 <= headline_metrics["macro_f1"] <= 1.0
    assert checkpoint_path.exists() and checkpoint_path.stat().st_size > 0
    assert scratch_checkpoint_path.exists() and scratch_checkpoint_path.stat().st_size > 0
    assert scratch_logits.shape == (len(test_dataset), NUM_CLASSES)
    assert set(benchmark_table["model"]) == {"ScratchResidualCNN", "EfficientNet-B0"}
    assert benchmark_table[["accuracy", "balanced_accuracy", "macro_f1"]].to_numpy().min() >= 0.0
    assert benchmark_table[["accuracy", "balanced_accuracy", "macro_f1"]].to_numpy().max() <= 1.0
    assert torchscript_path.exists() and torchscript_path.stat().st_size > 0
    assert onnx_path.exists() and onnx_path.stat().st_size > 0
    cpu_sample = inference_sample.cpu()
    test_output = traced_model(cpu_sample)
    reference_output = export_model(cpu_sample)
    assert torch.allclose(test_output, reference_output, atol=1e-4, rtol=1e-3)
    print("All VisionForge unit and invariant tests passed.")


run_unit_tests()

history_csv = ARTIFACT_DIR / "training_history.csv"
training_history.to_csv(history_csv, index=False)
report_csv = ARTIFACT_DIR / "class_metrics.csv"
report.to_csv(report_csv)
corruption_csv = ARTIFACT_DIR / "corruption_report.csv"
corruption_report.to_csv(corruption_csv, index=False)
policy_csv = ARTIFACT_DIR / "selective_policy.csv"
selective_curve.to_csv(policy_csv, index=False)

model_card = {
    "model_name": "VisionForge EfficientNet-B0",
    "dataset": CFG.dataset_id,
    "task": "Three-class bean-leaf image classification with abstention",
    "metrics": {**headline_metrics, **{f"policy_{k}": v for k, v in selected_policy.items()}},
    "controlled_benchmark": benchmark_table.to_dict(orient="records"),
    "intended_use": "Educational field-image triage prototype",
    "not_for": ["autonomous pesticide decisions", "medical use", "species outside the dataset"],
    "known_limitations": [
        "Small dataset and limited geography can create domain shift.",
        "Confidence calibration may drift under new cameras or seasons.",
        "Grad-CAM is a sensitivity visualisation, not a causal explanation.",
        "Corruption tests are proxies and do not cover every field condition.",
    ],
    "human_oversight": "Low-confidence images and treatment decisions require an agronomist.",
}
model_card_path = save_json(model_card, ARTIFACT_DIR / "model_card.json")
artifacts = [
    checkpoint_path, scratch_checkpoint_path, torchscript_path, onnx_path, model_card_path,
    benchmark_csv, benchmark_figure,
    history_csv, report_csv, corruption_csv, policy_csv,
    sample_figure, diagnostics_figure, policy_figure,
    stress_figure, explainability_figure,
]
manifest = make_manifest("VisionForge", asdict(CFG), headline_metrics, artifacts)
manifest_path = save_json(manifest, ARTIFACT_DIR / "manifest.json")
display(pd.DataFrame(manifest["artifacts"]))
print("manifest:", manifest_path)


All VisionForge unit and invariant tests passed.


,path,bytes,sha256
0,visionforge_artifacts/visionforge_state_dict.pt,17645563,aabdf7f94e82407b38da6d555c91e0cb0833641693fbb6...
1,visionforge_artifacts/visionforge_scratch_cnn_...,2131077,2c66fc31befa65b60f0a06f59cd9b02c6495c087b9945c...
2,visionforge_artifacts/visionforge_efficientnet.ts,18210362,d728abb554c8f26978f8c569f66ceec53e5fdc9ec581c0...
3,visionforge_artifacts/visionforge_efficientnet...,657128,b8026a4e837536c9036c45bbb4808abb7354b73305968b...
4,visionforge_artifacts/model_card.json,1847,177834b7245a0f35a2758e804e66d98946b36373ea8a65...
5,visionforge_artifacts/model_benchmark.csv,299,68285ab961a2134da4af201504168d4ec306b09ea49d83...
6,visionforge_artifacts/cnn_transfer_benchmark.png,47593,59c0cafc3a801a5de601106fa6a1e3578eb4e37c6f98ac...
7,visionforge_artifacts/training_history.csv,1437,b13f4ca3438a3d3abd33597c55957069fb15152ebf65cf...
8,visionforge_artifacts/class_metrics.csv,441,4d951a49793779167f97781f3c44fa72d44edfe72ddc16...
9,visionforge_artifacts/corruption_report.csv,1023,525b6f725979ca6d9fdf25675a9e363eb9451ed4158bb1...


manifest: visionforge_artifacts/manifest.json


## Interview-ready explanation

**Problem.** Field images vary in lighting, scale and background, and a wrong automated
label can lead to a costly intervention. I framed the system as selective prediction rather
than forcing a class for every image.

**Technical choice.** I first trained a custom residual CNN from scratch as a controlled baseline. I then transferred ImageNet features into EfficientNet, warmed up the new
head, selectively unfroze the final blocks, calibrated logits with validation-only
temperature scaling, and selected an abstention threshold before opening the test set.

**Evidence beyond accuracy.** The release includes per-class metrics, a reliability diagram,
accuracy–coverage analysis, image-corruption stress tests, Grad-CAM sensitivity maps,
exported models and inference latency. That makes failure behaviour visible.

**CV bullet after execution:**

> Built a trustworthy PyTorch vision system on 1,295 real field images, benchmarking a custom
> residual CNN against EfficientNet transfer learning with mixed precision and staged fine-tuning; added temperature calibration,
> uncertainty-based human escalation, Grad-CAM, corruption stress tests, ONNX/TorchScript
> export and an interactive image-inspection app.
